# 1.1 Precursor Producibility 

Produces a binary matrix, if a biomass precursor can be produced = 1 or not produced = 0

In [8]:
import os
import glob
import pandas as pd
from cobra.io import read_sbml_model
from cobra import Reaction
import copy

In [9]:
# Set the path to your folder with all models (change as needed)
models_folder = "/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass"
output_csv = "/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Biomass_Viability/Precursor_Produciblity/precursor_producibility_matrix_new.csv"
media_csv = "/scratch/rebernig/VR002_Model_Analysis/Scripts/Elenas_models/Transcriptomics_integration/Media_LB_minus_02/LB_minus_O2_media.csv"

In [ ]:
#This will define the function used later
def apply_media_by_metabolite_map(model, media_map, anaerobic=True, atpm_lb=5.54, verbose=False):
    """
    media_map: dict like {"glc__D":10, "glc__D_e":10, "nh4":1000, ...}
    Matches against each exchange's extracellular metabolite id (e.g., 'glc__D_e').
    If key without '_e' is present, it will also match.
    """
    med_mod = {ex.id: 0.0 for ex in model.exchanges}  # start closed
    opened = 0

    for ex in model.exchanges:
        met = next(iter(ex.metabolites.keys()))
        met_id = met.id            # e.g., "glc__D_e"
        base_id = met_id[:-2] if met_id.endswith("_e") else met_id  # "glc__D"
        cap = None
        if met_id in media_map:
            cap = float(media_map[met_id])
        elif base_id in media_map:
            cap = float(media_map[base_id])
        if cap is not None and cap > 0:
            med_mod[ex.id] = cap
            opened += 1

    # anaerobic
    if anaerobic and "EX_o2_e" in med_mod:
        med_mod["EX_o2_e"] = 0.0

    model.medium = med_mod

    # ATP maintenance
    if atpm_lb is not None and "ATPM" in model.reactions:
        model.reactions.ATPM.lower_bound = float(atpm_lb)

    if verbose:
        print(f"Applied medium; opened {opened} exchanges.")

In [11]:
LB_minus_O2_media = pd.read_csv(media_csv, index_col=0).squeeze().to_dict()

In [12]:
# ---------- find models ----------
model_files = glob.glob(os.path.join(models_folder, "*.xml"))
print(model_files)

# ---------- compute precursor producibility ----------
all_precursors = set()
model_precursor_results = {}

for model_file in model_files:
    print(f"Processing model: {model_file}")
    try:
        model = read_sbml_model(model_file)

        # >>> APPLY MEDIUM HERE <<<
        apply_media_by_metabolite_map(model, LB_minus_O2_media, anaerobic=True, atpm_lb=5.54, verbose=True)

        # biomass reaction
        biomass_rxn = model.reactions.get_by_id("Biomass")
        precursors = [met.id for met in biomass_rxn.reactants]
        all_precursors.update(precursors)

        # test each precursor
        results = {}
        for met_id in precursors:
            test_model = copy.deepcopy(model)  # keeps the applied medium
            demand = Reaction("DM_" + met_id)
            demand.add_metabolites({test_model.metabolites.get_by_id(met_id): -1})
            test_model.add_reactions([demand])
            test_model.objective = demand
            flux = test_model.slim_optimize()
            results[met_id] = int(flux is not None and flux >= 1e-6)

        model_name = os.path.basename(model_file)
        model_precursor_results[model_name] = results

    except Exception as e:
        print(f"Error processing {model_file}: {e}")

['/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/assembly.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/agora.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/core4.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/modelseed.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/core2.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/kbase.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/core5.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Confidence_Levels_Biomass/core3.xml', '/scratch/rebernig/VR002_Model_Analysis/Output/C

In [13]:
all_precursors = sorted(all_precursors)
matrix = pd.DataFrame(index=all_precursors)
for model_name, result in model_precursor_results.items():
    matrix[model_name] = [result.get(met, 0) for met in matrix.index]

matrix.to_csv(output_csv)
print(f"Results saved to {output_csv}")


Results saved to /scratch/rebernig/VR002_Model_Analysis/Output/Curated_Models_Elena/B.Uniformis/Biomass_Viability/Precursor_Produciblity/precursor_producibility_matrix_new.csv
